In [1]:
import numpy as np
import pandas as pd
import glob

In [2]:
files = glob.glob(r"F:\doc-hub\education\psychology\projects\research\stroop-interference\data\raw\*.csv")

dfs = []

for f in files:
    df = pd.read_csv(f, skiprows=1)
    df.columns = df.columns.str.strip()
    dfs.append(df)

df = pd.concat(dfs, ignore_index=True)

print(df.head())
print("\nColumns:\n", df.columns)
print("\nInfo:\n", df.info())

     subj  trial      time  color    condition  correct response        RT
0  118340      0  1.814140  green    congruent  correct        f  0.944198
1  118340      1  3.835822  green  incongruent  correct        f  0.794779
2  118340      2  5.855570    red  incongruent  correct        d  0.566762
3  118340      3  7.875476   blue      neutral  correct        j  0.674958
4  118340      4  9.895955  green      neutral  correct        f  1.118514

Columns:
 Index(['subj', 'trial', 'time', 'color', 'condition', 'correct', 'response',
       'RT'],
      dtype='str')
<class 'pandas.DataFrame'>
RangeIndex: 51000 entries, 0 to 50999
Data columns (total 8 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   subj       51000 non-null  int64  
 1   trial      51000 non-null  int64  
 2   time       51000 non-null  float64
 3   color      51000 non-null  str    
 4   condition  51000 non-null  str    
 5   correct    51000 non-null  str    
 6   respons

In [3]:
# Validation the strucute.

# Types and shape
print("Dtypes:\n", df.dtypes)
print("\nData shape: ", df.shape)

# Number of trials per condition
print("\nNumber of trials:\n", df["condition"].value_counts())

# Number of unique participants
print("\nParticipants: ", df["subj"].nunique())

# Missing values
print("\nMissing values:\n", df.isnull().sum())

Dtypes:
 subj           int64
trial          int64
time         float64
color            str
condition        str
correct          str
response         str
RT           float64
dtype: object

Data shape:  (51000, 8)

Number of trials:
 condition
congruent      16449
incongruent    16447
neutral        16413
probe           1691
Name: count, dtype: int64

Participants:  81

Missing values:
 subj         0
trial        0
time         0
color        0
condition    0
correct      0
response     0
RT           0
dtype: int64


Note: The conditions are approximately  balanced across core Stroop conditions; although, there is a Small differences (<1%). There are 81 unique participants. It seems that there is no missing values.

In [4]:
# Checking RT.

df["RT"] = pd.to_numeric(df["RT"], errors="coerce")

print(df.groupby("condition")["RT"].describe())

               count      mean       std  min       25%       50%       75%  \
condition                                                                     
congruent    16449.0  0.655516  0.279417 -1.0  0.520361  0.613710  0.763355   
incongruent  16447.0  0.731579  0.386374 -1.0  0.562673  0.712772  0.935480   
neutral      16413.0  0.670495  0.284806 -1.0  0.529153  0.634222  0.790996   
probe         1691.0 -1.000000  0.000000 -1.0 -1.000000 -1.000000 -1.000000   

                  max  
condition              
congruent    1.597732  
incongruent  1.599897  
neutral      1.596369  
probe       -1.000000  


Note: -1 values likely represent invalid or missing responses.

In [5]:
# Changing -1 to nan.

df["RT"] = pd.to_numeric(df["RT"], errors="coerce")
df = df[df["RT"] > 0]

# Removing invalid condition.

df = df[df["condition"] != "probe"]

# Recomputing summaries.

print(df.groupby("condition")["RT"].describe())

               count      mean       std       min       25%       50%  \
condition                                                                
congruent    16272.0  0.673524  0.220872  0.012440  0.523763  0.616308   
incongruent  16020.0  0.777733  0.266852  0.025451  0.574043  0.721489   
neutral      16225.0  0.689851  0.222134  0.011215  0.532954  0.636727   

                  75%       max  
condition                        
congruent    0.766082  1.597732  
incongruent  0.943960  1.599897  
neutral      0.793167  1.596369  


In [6]:
# Saveing merged dataset.

df.to_csv(r"F:\doc-hub\education\psychology\projects\research\stroop-interference\data\processed\stroop-merged.csv", index=False)

### Dataset Dictionary

#### subj
- meaning: participant identifier
- role: grouping variable (between-subject factor)
- type: categorical (integer-coded)
- notes: 81 unique participants; repeated-measures structure within each subject

#### trial
- meaning: trial index within participant
- role: within-subject sequencing variable
- type: integer (ordinal)
- notes: resets per participant; not globally unique

#### time
- meaning: elapsed time during experiment (trial onset or timestamp)
- role: temporal tracking variable
- type: continuous numeric (float)
- units: seconds

#### color
- meaning: stimulus ink color presented on each trial
- role: experimental stimulus feature
- type: categorical (nominal)
- values: red, green, blue, yellow (observed set)

#### condition
- meaning: Stroop experimental condition
- role: independent variable (within-subject factor)
- type: categorical (nominal)
- values: congruent, incongruent, neutral, probe

#### correct
- meaning: response accuracy label
- role: performance outcome variable
- type: categorical (binary-like)
- values: correct, incorrect

#### response
- meaning: participant keypress response
- role: behavioral response variable
- type: categorical

#### RT
- meaning: reaction time from stimulus onset to response
- role: dependent variable (primary outcome)
- type: continuous numeric
- units: seconds
- notes: cleaned from invalid values (-1 removed as NaN)